O coronavírus pegou o mundo todo de surpresa mudando a rotina das pessoas. Os moradores das cidades já não passavam mais seu tempo livre fora de casa, indo a cafés, e shoppings; a maioria ficou em casa lendo livros. Isso chamou a atenção de startups que se apressaram para desenvolver novos aplicativos para os amantes de livros.

Você recebeu um banco de dados de um dos serviços concorrentes nesse mercado. Ele contém dados sobre livros, editoras, autores, e classificação de clientes e avaliação de livros. Essa informação será usada para gerar uma proposição válida para o novo produto.

### Descrição dos dados

**`books` — livros:**

Contém dados sobre livros:

- `book_id` — identificador do livro
- `author_id` — identificador do autor
- `title` — título
- `num_pages` — número de páginas
- `publication_date` — data de publicação
- `publisher_id` — identificador da editora

**`authors` — autores:**

Contém dados sobre os autores:

- `author_id` — identificador do autor
- `author` — autor

**`publishers` — editoras:**

Contém dados sobre editoras:

- `publisher_id` — identificador da editora
- `publisher` — editora

**`ratings` — classificações:**

Contém dados sobre classificação dos usuários:

- `rating_id` — identificador da classificação
- `book_id` — identificador do livro
- `username` — o nome do usuário que avaliou o livro
- `rating` — classificação

**`reviews` — avaliação:**

Contém dados sobre revisão dos clientes:

- `review_id` — identificador da revisão
- `book_id` — identificador do livro
- `username` — o nome do usuário que revisou o livro
- `text` — o texto da revisão

### Conectando no banco de dados

In [1]:
import pandas as pd
from sqlalchemy import create_engine


db_config = {'user': 'practicum_student',  # nome de usuário
             'pwd': 's65BlTKV3faNIGhmvJVzOqhs', # senha
             'host': 'rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net',
             'port': 6432,              # porta de conexão
             'db': 'data-analyst-final-project-db'}          # o nome do banco de dados

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(db_config['user'],
                                                                     db_config['pwd'],
                                                                       db_config['host'],
                                                                       db_config['port'],
                                                                       db_config['db'])

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

# <center> Tarefa </center>

### Encontre o número de livros lançados depois de 1 de janeiro de 2000.

Vamos criar uma função para facilitar a consulta sempre que precisarmos

In [2]:
def consult_sql(query):
    consult = pd.io.sql.read_sql(query, con = engine)
    return consult

Dando uma olhada no conjunto de dados 'books'

In [3]:
books = consult_sql('SELECT * FROM books')

display(books.head())

,book_id,author_id,title,num_pages,publication_date,publisher_id
0,1,546,'Salem's Lot,594,2005-11-01,93
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309
4,5,125,1776,386,2006-07-04,268


In [4]:
count_books = consult_sql(
    '''
    SELECT 
    COUNT(title) as book_count
    FROM books 
    WHERE publication_date > '2000-01-01'
    '''
    )

print('Número de livros lançados após 01/01/2000:'  )
print(count_books)

Número de livros lançados após 01/01/2000:
   book_count
0         819


Foram lançados 819 livros após dia 01/01/2000

### Encontre o número de avaliações e a classificação média para cada livro.

Vamos dar uma olhada na tabela 'ratings':

In [5]:
rating = consult_sql(
    '''
SELECT 
*
FROM ratings
    '''
)

display(rating.head())

,rating_id,book_id,username,rating
0,1,1,ryanfranco,4
1,2,1,grantpatricia,2
2,3,1,brandtandrea,5
3,4,2,lorichen,3
4,5,2,mariokeller,2


Contando o número de avaliações

In [21]:
mean_rating = consult_sql(
'''
SELECT
    books.title,
    subquery.avg_rating,
    subquery.review_count
FROM
    (SELECT
         reviews.book_id AS book_id,
         COUNT(DISTINCT reviews.review_id) AS review_count,
         floor(AVG(ratings.rating)) AS avg_rating
     FROM
         reviews
         INNER JOIN ratings ON ratings.book_id = reviews.book_id
     GROUP BY
         reviews.book_id
     ORDER BY
         review_count DESC,
         avg_rating DESC) AS subquery
    INNER JOIN books ON subquery.book_id = books.book_id
ORDER BY
    review_count DESC,
    avg_rating DESC
'''

)

display(mean_rating.head(10))

,title,avg_rating,review_count
0,Twilight (Twilight #1),3.0,7
1,The Glass Castle,4.0,6
2,The Book Thief,4.0,6
3,Harry Potter and the Chamber of Secrets (Harry...,4.0,6
4,Harry Potter and the Prisoner of Azkaban (Harr...,4.0,6
5,Outlander (Outlander #1),4.0,6
6,The Curious Incident of the Dog in the Night-Time,4.0,6
7,The Lightning Thief (Percy Jackson and the Oly...,4.0,6
8,The Hobbit or There and Back Again,4.0,6
9,The Road,3.0,6


Os livros que mais teve avaliações foi o Crepúsculo, com 7 avaliações e uma nota média de 3.0.

In [1]:
# query_t2 = """SELECT
#               books.title,
#               subquery.avg_rating,
#               subquery.review_count
#               FROM
#                   (SELECT
#                        reviews.book_id as book_id,
#                        COUNT (DISTINCT reviews.review_id) AS review_count,
#                        AVG (ratings.rating) AS avg_rating
#                    FROM
#                        reviews
#                        INNER JOIN ratings ON ratings.book_id = reviews.book_id
#                    GROUP BY
#                        reviews.book_id
#                    ORDER BY
#                        review_count DESC,
#                        avg_rating DESC) AS subquery
#                    INNER JOIN books ON subquery.book_id = books.book_id
#               ORDER BY
#                   review_count DESC,
#                   avg_rating DESC
#               LIMIT 10"""

### Identifique a editora que lançou o maior número de livros com mais de 50 páginas (isso vai ajudar você a excluir brochuras e publicações parecidas da sua análise).

Vamos consultar as primeiras linhas da tabela para entender

In [84]:
publisheres = consult_sql(
'''
SELECT * FROM publishers
'''
)

display(publisheres.head())

,publisher_id,publisher
0,1,Ace
1,2,Ace Book
2,3,Ace Books
3,4,Ace Hardcover
4,5,Addison Wesley Publishing Company


In [57]:
publisher_50 = consult_sql(
    '''
SELECT 
publishers.publisher,
(COUNT(publishers.publisher)) AS count
FROM publishers INNER JOIN books on publishers.publisher_id  = books.publisher_id
WHERE books.num_pages > 50
GROUP BY publishers.publisher
ORDER BY count DESC
    '''

)

display(publisher_50.head())

,publisher,count
0,Penguin Books,42
1,Vintage,31
2,Grand Central Publishing,25
3,Penguin Classics,24
4,Ballantine Books,19


A editora Penguin Books foi a que mais lançou livros com mais de 50 páginas, sendo 42 livros lançados, seguido por Vintage com 31 e Grand Central Publishing, com 25.

### Identifique o autor com a média mais alta classificação de livros: olhe apenas para livros com pelo menos 50 classificações.

Vamos consultar as primeiras linhas da tabela para entender

In [86]:
ratings = consult_sql(
'''
SELECT * from ratings
'''
)

display(ratings.head())

,rating_id,book_id,username,rating
0,1,1,ryanfranco,4
1,2,1,grantpatricia,2
2,3,1,brandtandrea,5
3,4,2,lorichen,3
4,5,2,mariokeller,2


In [23]:
author_rating = consult_sql(
    '''
SELECT
    authors.author,
    AVG(subquery2.avg_rating) AS final_avg            
FROM
    (SELECT
        books.title,
        books.author_id,
        subquery1.avg_rating
    FROM
        (SELECT
            book_id,
            COUNT(rating_id) AS rating_count,
            floor(AVG(rating)) AS avg_rating
        FROM
            ratings
        GROUP BY
            book_id
        HAVING
            COUNT(rating_id) > 50) AS subquery1
        INNER JOIN books ON books.book_id = subquery1.book_id) AS subquery2
    INNER JOIN authors ON authors.author_id = subquery2.author_id
GROUP BY
    authors.author
ORDER BY
    final_avg DESC
    '''
)

display(author_rating.head(5))

,author,final_avg
0,J.K. Rowling/Mary GrandPré,4.0
1,Rick Riordan,4.0
2,Markus Zusak/Cao Xuân Việt Khương,4.0
3,Louisa May Alcott,4.0
4,J.R.R. Tolkien,4.0


Na lista dos autores com maiores classificações, temos nomes como J.K Rowling e Rick Riordan, J.R.R Tolkien, escritores de série de livros clássicos e que fazem muito sucesso até hoje

In [24]:
# query4= """SELECT
#                   authors.author,
#                   AVG (subquery2.avg_rating) as final_avg            
#               FROM
#                   (SELECT
#                       books.title,
#                       books.author_id,
#                       subquery1.avg_rating
#                   FROM
#                       (SELECT
#                           book_id,
#                           COUNT (rating_id) AS rating_cnt,
#                           AVG (rating) AS avg_rating
#                       FROM
#                           ratings
#                       GROUP BY
#                           book_id
#                       HAVING
#                           COUNT (rating_id) > 50) AS subquery1
#                       INNER JOIN books ON books.book_id = subquery1.book_id) AS subquery2
#                   INNER JOIN authors ON authors.author_id = subquery2.author_id
#               GROUP BY
#                   author
#               ORDER BY
#                   final_avg DESC
#               LIMIT 5"""

### Encontre o número médio de avaliações entre usuários que classificaram mais do que 50 livros.

Vamos verificar os avaliadores:

In [87]:
reviews = consult_sql(
    '''
SELECT * from reviews 
    '''
)

display(reviews.head())


,review_id,book_id,username,text
0,1,1,brandtandrea,Mention society tell send professor analysis. ...
1,2,1,ryanfranco,Foot glass pretty audience hit themselves. Amo...
2,3,2,lorichen,Listen treat keep worry. Miss husband tax but ...
3,4,3,johnsonamanda,Finally month interesting blue could nature cu...
4,5,3,scotttamara,Nation purpose heavy give wait song will. List...


In [29]:
count_reviews = consult_sql(
'''
SELECT
                  floor(AVG (subquery2.review_count)) AS avg_review_count
              FROM
                  (SELECT
                      COUNT (reviews.review_id) as review_count,
                      subquery1.username
                  FROM
                      (SELECT
                          username,
                          COUNT (rating_id) AS rating_count
                      FROM
                          ratings
                      GROUP BY
                          username
                      HAVING
                          COUNT (rating_id) > 50) AS subquery1
                      INNER JOIN reviews ON reviews.username = subquery1.username
                  GROUP BY
                      subquery1.username) AS subquery2
'''
)

display(count_reviews.head())

,avg_review_count
0,24.0
